# Prisoner's Dilemma Basics

We create a model based on agents which are playing the Prisoner's Dilemma game against eachother. 
The payoff of each game is determined as below: 


<img src="../Project/payoff.jpg" />

## Agent Definition

We define each agent with 2 parameters **probability of defection** and **memory size** (each memory slot is about one other agent)
An agent also has 3 attributes of the **score** the agent has accumulated throughout playing and the seperate slots of **memory** for cooperating and defecting agents. 
The supposed goal of the agents are to obtain the highest score so we can say an agent with a higher score is better.


In [6]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline  
import pandas as pd
import random

#Memory cells for storing info about agents past games

class memory_cell():
    def __init__(self, act = 'Coop'):
        self.defect_number_ = 0.0
        self.play_number_ = 1.0
        
        if act == 'Defect' : self.defect_number_ += 1.0
    
    def update(self, act):
        if act == 'Defect' : self.defect_number_ += 1.0
        self.play_number_ += 1.0
    
    def impression(self):
        #The evaluation of the defect/play ratio of the agent in memory
        return self.defect_number_/max(1.0, self.play_number_)

# Agent class that plays the game
    
class agent():
 
    def __init__(self, ID, pd = 1, M = 1):
        self.ID = ID
        self.pd = pd
        self.capacity = M
        self.score_ = 0
        
        self.memory_for_defectors_ = dict()
        self.memory_for_cooperators_ = dict()
        
    def act(self):
        # Return defection or cooperation based on the agents probability of defection attribute
        if np.random.rand() < self.pd:
            return 'Defect'
        return 'Coop'
    
    def know(self, other):
        #Checks if the current agent has the other agent in memory
        return (other in self.memory_for_defectors_) or (other in self.memory_for_cooperators_)
    
    def is_memory_full(self):
        #Checks if memory is full
        c,d = len(self.memory_for_cooperators_), len(self.memory_for_defectors_)
        return  c + d >= self.capacity
    
    def perceive_as_defector(self, other):
        #Checks if another agent is a defector
        return other in self.memory_for_defectors_
    
    def perceive_as_cooperator(self, other):
        #Checks if another agent is a cooperator
        return other in self.memory_for_cooperators_
    
    def add_payoff(self, payoff):
        #Add a payoff to agents score value
        self.score_ += payoff
    
    def learn(self, other, act):
        #The function for playing against another agent and adding them to memory
        if perceive_as_defector(self, other):
            print("defector- don't play")
        elif perceive_as_cooperator(self, other):
            self.memory_for_cooperators_[other].update(act)
            
            # Checks the evaluation of cooperator and if ratio is bigger than 1/2 for defection transfers it to defector memory
            if self.memory_for_cooperators_[other].impression() > 0.5:
                self.memory_for_defectors_[other] = self.memory_for_cooperators_[other]
                del self.memory_for_cooperators_[other]
        else: 
            # Plays with an unknown agent
            if self.is_memory_full(): 
                self.forget()
            if act == 'Coop':
                self.memory_for_cooperators_[other] = memory_cell(act)
            else:
                self.memory_for_defectors_[other] = memory_cell(act)
        
    def forget(self):  
        #Deleting an agent from memory randomly
        if len(self.memory_for_cooperators_) > 0:
            f = random.choice(list(self.memory_for_cooperators_.keys()))
            del self.memory_for_cooperators_[f]
        else:
            f = random.choice(list(self.memory_for_defectors_.keys()))
            del self.memory_for_defectors_[f]

    def display(self):
        #For showing as table
        return [(k.ID, v.impression()) for (k,v) in self.memory_for_cooperators_.items()] + [(k.ID, v.impression()) for (k,v) in self.memory_for_defectors_.items()]

## The Model

Now we have the agents to play the game but we are required to create the model that the agents will interact with eachother. The parameters for the model are as follows: **number of agents** in the model, **memory size** for all agents in the model, the **population** that contains the features of each agent, the **time** variable to determine the amount of games played in the model and finally the **payoff matrix** of the game to be used(this is shown in the image above).

In [ ]:
class abm():
    def __init__(self, N = 10, M = 9, tau = 30, payoff = {'CC':3,'CD':0,'DC':5,'DD':1}):
        self.N = N
        self.time = self.N * self.N * tau
        self.population = [agent(i, pd = np.random.rand(), M = M) for i in range(self.N)]
        self.payoff = payoff
        
    def world(self):
        for i in range(self.time):
            iA, iB = np.random.choice(range(self.N), 2, replace=False)
            A, B = self.population[iA], self.population[iB]
            
            # Decision to interact
            if B.perceive_as_defector(A): continue
            if A.perceive_as_defector(B): continue
            
            A_action, B_action = A.act(), B.act()
            A.learn(B, B_action)
            B.learn(A, A_action)

            A.take_payoff(self.payoff[A_action[:1]+B_action[:1]])
            B.take_payoff(self.payoff[B_action[:1+A_action[:1]])
            
    def display(self):
        df = pd.DataFrame(columns=['ID','pD', 'score', 'M', 'Memory Len', 'Memory'])
        for A in self.population:
            df.loc[A.ID] = pd.Series({'ID':A.ID, 'pD':A.pd, 'score':A.score_, 'M': A.capacity, 
                                      'Memory Len': len(A.memory_for_cooperators_) + len(A.memory_for_defectors_), 
                                      'Memory':A.display()})
        return df